In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [ ]:
!pip install voila

In [12]:
# ==========================================
# 1. MEMBUAT ELEMEN GAMBAR (DI SEBELAH KANAN)
# ==========================================
# Ganti "petunjuk.png" di bawah ini dengan nama file gambar Anda yang sebenarnya
try:
    with open("alterasi-tapselA1.jpg.jpeg", "rb") as file:
        image_bytes = file.read()
    
    gambar_widget = widgets.Image(
        value=image_bytes,
        format='png', # ubah ke 'jpg' jika gambar Anda berformat jpg
        width=600,    # Sesuaikan ukuran lebar gambar
        layout=widgets.Layout(margin='0px 0px 0px 0px') # Jarak dari sisi kiri
    )
except FileNotFoundError:
    # Jika gambar belum ada/salah nama, tampilkan placeholder tulisan agar kode tidak error
    gambar_widget = widgets.HTML(
        value="<div style='border: 2px dashed #ccc; padding: 50px; text-align: center; width: 400px; margin-left: 20px;'>"
              "🖼️ <b>Gambar Tidak Ditemukan</b><br>"
              "Pastikan file gambar berada di folder yang sama dengan file notebook ini "
              "dan namanya sudah sesuai di dalam kode.</div>"
    )

# ==========================================
# 2. ELEMEN INPUT DAN OUTPUT (DI SEBELAH KIRI)
# ==========================================
dropdown_style = {'description_width': 'initial'}

html_title = widgets.HTML(
    value="<h2>🌋 Identifikasi Penentuan Zonasi & Potensi REE</h2>"
          "<h4>Sistem Pakar Wilayah Tapanuli Selatan</h4><hr>"
)

ph_widget = widgets.Dropdown(
    options=['-- Pilih Parameter --', 'Sangat Asam (pH < 4)', 'Asam Lemah–Netral (pH 4–5)', 'Netral–Basa (pH 6–8)', 'Oksidatif'],
    value='-- Pilih Parameter --',
    description='1. Pilih pH Fluida:',
    style=dropdown_style,
    layout=widgets.Layout(width='450px')
)

suhu_widget = widgets.Dropdown(
    options=['-- Pilih Parameter --', 'Suhu Tinggi (200°C–350°C)', 'Suhu Sedang (100°C–200°C)', 'Suhu Sedang (200°C–300°C)', 'Suhu Permukaan < 50°C'],
    value='-- Pilih Parameter --',
    description='2. Pilih Suhu Fluida:',
    style=dropdown_style,
    layout=widgets.Layout(width='450px')
)

kurva_widget = widgets.Dropdown(
    options=['-- Pilih Parameter --', 'Depresi / Defisit', 'Pengayaan LREE', 'Pola Normal', 'Anomali Ce Positif'],
    value='-- Pilih Parameter --',
    description='3. Pilih Pola Kurva REE:',
    style=dropdown_style,
    layout=widgets.Layout(width='450px')
)

button_run = widgets.Button(
    description='🚀 JALANKAN ANALISIS',
    button_style='primary',
    layout=widgets.Layout(width='450px', margin='15px 0px 15px 0px')
)

# Area khusus hasil diagnosis
output_area = widgets.Output()

# ==========================================
# 3. LOGIKA SISTEM PAKAR
# ==========================================
def on_button_clicked(b):
    with output_area:
        clear_output()
        
        pH = ph_widget.value
        suhu = suhu_widget.value
        kurva = kurva_widget.value
        
        if pH == '-- Pilih Parameter --' or suhu == '-- Pilih Parameter --' or kurva == '-- Pilih Parameter --':
            display(HTML("<div style='color: #c0392b; font-weight: bold; padding: 10px; background-color: #fce4d6; border-radius: 5px; width: 450px;'>"
                         "⚠️ Mohon lengkapi semua parameter input terlebih dahulu!</div>"))
            return
            
        zona = ""
        mobilitas = ""
        mineral = ""
        potensi = ""
        bg_color = ""
        flag_match = False
        
        # Opsi A: Advanced Argilik
        if "Sangat Asam" in pH and "Depresi" in kurva:
            zona = "Advanced Argilik"
            mobilitas = "Sangat Mobil (Terlindi): Cairan asam kuat melarutkan REE dari batuan asal. REE terbawa menjauhi inti sistem."
            mineral = "Minimal. Sebagian kecil LREE terikat di dalam struktur mineral Alunita."
            potensi = "RENDAH: Zona ini kaya emas-tembaga, tetapi miskin LTJ karena pencucian geokimia yang masif."
            bg_color = "#f8d7da"
            flag_match = True
            
        # Opsi B: Argilik
        elif "Asam Lemah" in pH and "Pengayaan LREE" in kurva:
            zona = "Argilik"
            mobilitas = "Imobil Sebagian (Terjebak): REE yang terlarut dari zona inti mulai mengendap kembali karena penurunan keasaman fluida."
            mineral = "Mineral lempung Ilit dan Smektit melalui proses adsorpsi ion pada permukaan lempung."
            potensi = "SEDANG: Mulai terjadi akumulasi REE jenis adsorpsi ion (ion-adsorption clay)."
            bg_color = "#fff3cd"
            flag_match = True
            
        # Opsi C: Propilitik
        elif "Netral–Basa" in pH and "Pola Normal" in kurva:
            zona = "Propilitik"
            mobilitas = "Imobil / Isokimia: REE cenderung stabil karena fluida tidak cukup asam untuk merusak struktur batuan secara total."
            mineral = "Epidot, Kalsit, dan Apatit sekunder pembawa REE hasil substitusi kalsium (Ca)."
            potensi = "RENDAH HINGGA SEDANG: REE tertahan di dalam kisi kristal mineral silikat primer/sekunder (sulit diekstraksi)."
            bg_color = "#e2e3e5"
            flag_match = True
            
        # Opsi D: Tudung Oksidasi
        elif "Oksidatif" in pH and "Anomali Ce" in kurva:
            zona = "Tudung Oksidasi (Supergen / Laterit)"
            mobilitas = "Imobilisasi & Pengayaan Residual: Pelapukan intensif melarutkan unsur mayor (Na, Ca, Mg), menyisakan REE yang tidak larut di permukaan."
            mineral = "Oksida besi (Geotit, Hematit) melalui penyerapan permukaan, serta sisa mineral resisten (Monasit, Xenotim)."
            potensi = "SANGAT TINGGI: Merupakan zona target utama untuk tipe endapan LTJ Laterit (paling ekonomis diolah)."
            bg_color = "#d1e7dd"
            flag_match = True

        if flag_match:
            output_html = f"""
            <div style="padding: 15px; border-radius: 8px; width: 450px; font-family: sans-serif; border: 1px solid #ccc; background-color: #ffffff;">
                <h3 style="margin-top: 0; color: #333;">📊 HASIL DIAGNOSIS SISTEM</h3>
                <div style="background-color: {bg_color}; padding: 10px; border-radius: 5px; margin-bottom: 10px;">
                    <strong>Zona Alterasi Terdeteksi:</strong><br>
                    <span style="font-size: 1.1em; font-weight: bold;">{zona}</span>
                </div>
                <p><strong>🔬 Karakteristik Mobilitas REE:</strong><br>{mobilitas}</p>
                <p><strong>🪨 Mineral Pembawa Utama:</strong><br>{mineral}</p>
                <p><strong>💎 Rekomendasi Potensi REE:</strong><br><span style="color: #0f5132; font-weight: bold;">{potensi}</span></p>
            </div>
            """
            display(HTML(output_html))
        else:
            display(HTML(
                "<div style='color: #842029; font-weight: bold; padding: 15px; background-color: #f8d7da; border-radius: 5px; width: 450px;'>"
                "❌ Data Tidak Sesuai!<br><span style='font-weight: normal; font-size: 0.9em;'>"
                "Kombinasi parameter tidak sesuai dengan model zonasi geokimia Tapanuli Selatan. Tinjau kembali data Anda.</span></div>"
            ))

button_run.on_click(on_button_clicked)

# ==========================================
# 4. MEMBUAT LAYOUT BERDAMPINGAN (HBOX)
# ==========================================
# Mengelompokkan panel kiri secara vertikal
panel_kiri = widgets.VBox([ph_widget, suhu_widget, kurva_widget, button_run, output_area])

# Menyatukan panel kiri dan gambar kanan secara horizontal
layout_aplikasi = widgets.HBox([panel_kiri, gambar_widget])

# Menampilkan semua ke layar Jupyter
display(html_title, layout_aplikasi)

HTML(value='<h2>🌋 Identifikasi Penentuan Zonasi & Potensi REE</h2><h4>Sistem Pakar Wilayah Tapanuli Selatan</h…